# HW 5 — From Scratch: Build a Diffusion Model (DDPM)
**Modern Computer Vision** — Technion, Spring 2026

**Due:** July 19, 2026

## Submission Requirements

**IMPORTANT:** Submit this notebook **with all outputs** — run all cells before submitting. Notebooks without outputs will lose points. Make sure your `STUDENT_ID` is set correctly.

**Student ID:**

In [ ]:
STUDENT_ID = ""  # <-- Fill in your student ID

---
## The idea

A **diffusion model** learns to turn noise into data by *reversing* a noising process:

- **Forward** (trivial, no learning): take a clean image and gradually stir in Gaussian noise until nothing is left but static.
- **Reverse** (the model): start from pure static and, step by step, *remove* a little noise — until an image appears.

The whole trick is a network that, given a noisy image and how noisy it is, predicts **the noise**. You'll implement the diffusion — the schedule, the forward jump, the training loss, and the sampler — in **four short functions**. The network itself (a small U-Net) is provided. This is the new flavour of *from scratch*: `torch.nn` and autograd are allowed; what you build is the **diffusion algorithm**.

*(If you did the theory question: the noise predictor you train here is exactly the learned **score** — the global quantity a single particle could never compute on its own.)*

By the end: pure noise → handwritten digits.

In [ ]:
# torch, torchvision and matplotlib come preinstalled on the course A10 (nothing to install)
import torch
import torch.nn.functional as F
from ddpm_utils import device, grab, UNet, get_mnist, train, show
print('device:', device)

---
## The data

MNIST digits, scaled to $[-1,1]$ (and padded to $32\times32$ for the U-Net):

In [ ]:
loader = get_mnist(batch_size=128, n=60000)   # n=None for the full 60k
batch, _ = next(iter(loader))
show(batch, 'real MNIST', ncol=8)

---
## Step 1 — The noise schedule

The forward process adds a little Gaussian noise at each of $T$ steps. The schedule sets how much: $\beta_t$ is the noise injected at step $t$, $\alpha_t = 1-\beta_t$, and $\bar\alpha_t = \prod_{s\le t}\alpha_s$ is the fraction of the **original signal** that survives to step $t$ — it slides from $\approx 1$ down to $\approx 0$.

In [ ]:
def make_schedule(T=1000):
    """The forward noise schedule. Returns a dict of (T,) tensors on `device`:
      beta  -- how much noise is injected at each step (small, increasing),
      alpha -- 1 - beta,
      abar  -- cumulative product of alpha == the fraction of the ORIGINAL signal
               still present at step t (starts ~1, ends ~0).
    Use a linear beta schedule from 1e-4 to 0.02.
    """
    # TODO -- your implementation here
    raise NotImplementedError("make_schedule")

In [ ]:
sched = make_schedule(T=1000)
print('abar starts at %.3f, ends at %.4f' % (sched['abar'][0], sched['abar'][-1]))
assert sched['abar'][0] > 0.99 and sched['abar'][-1] < 0.02, 'abar should go ~1 -> ~0'
import matplotlib.pyplot as plt
plt.figure(figsize=(5,2.6)); plt.plot(sched['abar'].cpu()); plt.xlabel('t'); plt.ylabel(r'$\bar\alpha_t$')
plt.title('signal retained vs. timestep'); plt.tight_layout(); plt.show()

---
## Step 2 — The forward process

Because the steps are Gaussian, you can jump from a clean image straight to **any** noise level in one shot:

$$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon,\qquad \epsilon\sim\mathcal N(0,I).$$

In [ ]:
def q_sample(x0, t, noise, sched):
    """Forward process: jump straight from a clean image x0 to its noised version
    at timestep t, in one shot:
        x_t = sqrt(abar_t) * x0  +  sqrt(1 - abar_t) * noise
    Args:  x0, noise : (B,1,32,32);  t : (B,) integer timesteps.
    Hint:  grab(sched['abar'], t) gives abar_t shaped (B,1,1,1) to broadcast.
    """
    # TODO -- your implementation here
    raise NotImplementedError("q_sample")

Watch one digit dissolve into noise as $t$ grows — your forward process in action:

In [ ]:
x0 = batch[:1].repeat(8, 1, 1, 1).to(device)
ts = torch.linspace(0, sched['T'] - 1, 8).long().to(device)
noised = q_sample(x0, ts, torch.randn_like(x0), sched)
show(noised, 't = 0  ......  T-1   (forward diffusion)', ncol=8)

---
## Step 3 — Train the noise predictor

Train the network on one job: given a noisy image $x_t$ and its timestep $t$, **predict the noise $\epsilon$** that was added. Corrupt each image to a random $t$, and minimise the MSE between the predicted and the true noise. That's all a diffusion model is trained to do.

In [ ]:
def ddpm_loss(model, x0):
    """The training objective. Corrupt each image to a RANDOM timestep, then ask
    the network to predict the exact noise that was added; the loss is the MSE
    between the predicted noise and the true noise. (Uses the global `sched`.)
    """
    # TODO -- your implementation here
    raise NotImplementedError("ddpm_loss")

Now optimise it (the `train` loop is provided). This runs 50 epochs over the full dataset — about 20–30 minutes on an A10 GPU (less on a bigger card):

In [ ]:
model = UNet().to(device)
losses = train(model, loader, ddpm_loss, epochs=50, lr=1e-3)
plt.figure(figsize=(5,2.6)); plt.plot(losses, lw=0.6); plt.xlabel('step'); plt.ylabel('loss')
plt.title('training loss'); plt.tight_layout(); plt.show()

---
## Step 4 — Sample: reverse the diffusion

Start from pure noise $x_T\sim\mathcal N(0,I)$ and walk **backwards**, $t=T\!-\!1,\dots,0$, each step nudging $x$ toward less-noisy using the predicted noise:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\Big(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon_\theta(x_t,t)\Big) + \sigma_t z,\qquad \sigma_t=\sqrt{\beta_t},\; z\sim\mathcal N(0,I)$$

(no noise is added on the final step).

In [ ]:
@torch.no_grad()
def sample(model, sched, n=64):
    """Reverse process: start from pure Gaussian noise and denoise step by step,
    from t = T-1 down to 0, using the model's noise prediction each step:
        mean      = (x_t - (1 - alpha_t)/sqrt(1 - abar_t) * eps) / sqrt(alpha_t)
        x_{t-1}   = mean + sigma_t * z ,   sigma_t = sqrt(beta_t),  z ~ N(0, I)
    (no noise is added on the very last step, t = 0). Returns (n,1,32,32) images.
    """
    # TODO -- your implementation here
    raise NotImplementedError("sample")

---
## Pure noise → digits 🎉

Run the sampler — this is your model dreaming up MNIST from scratch:

In [ ]:
samples = sample(model, sched, n=64)
show(samples, 'generated by your diffusion model', ncol=8)

---
*Modern Computer Vision — Technion — Spring 2026*